In [28]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch
import torch.nn.functional as F
from random import randint
import numpy as np

In [113]:
class LiveRecommendationDataset(Dataset):
    def __init__(self, data_file, hash_size=100000): 
        self.data = pd.read_csv(data_file, sep='\x01', header=None, na_values='\\N').fillna(0)

        self.features = {
            'cur_features': {
                'common_signs': list(self.data.iloc[:, 71]),  # 71
                'item_signs': list(self.data.iloc[:, 75]),   # 75
                'common_slots': list(self.data.iloc[:, 70]),  # 70
                'item_slots': list(self.data.iloc[:, 74]),   # 74
                'user_type': list(self.data.iloc[:, 15])      # 15
            },
            'nxt_features': {
                'common_signs': list(self.data.iloc[:, 73]),  # 73
                'item_signs': list(self.data.iloc[:, 77]),   # 77
                'common_slots': list(self.data.iloc[:, 72]),  # 72
                'item_slots': list(self.data.iloc[:, 76]),   # 76
                'user_type': list(self.data.iloc[:, 35])      # 25
            }
        }

        self.labels = {
            "cur_labels":{
                'live_end_auto_watch': list(self.data.iloc[:, 45]), # 45
                'live_end_watch': list(self.data.iloc[:, 47]), # 47
                'video_play_time': list(self.data.iloc[:, 17]), # 17
                'video_play_cnt': list(self.data.iloc[:, 18]), # 18
                'reco_ban': list(self.data.iloc[:, 66]), # 66
                ### 真正的label信息定义如下：
                'time_ratio': [],
                'time_reward': [],
                'time_idx_vec': [],
                'time_reward_start': [],
                'time_reward_range': [],
                'time_delta': [],
                
                'photo_ratio': [],
                'photo_reward': [],
                'photo_idx_vec': [],
                'photo_reward_start': [],
                'photo_reward_range': []
            },
            "nxt_labels":{
                'live_end_auto_watch': list(self.data.iloc[:, 84]), # 84
                'live_end_watch': list(self.data.iloc[:, 86]), # 86
                'video_play_time': list(self.data.iloc[:, 90]), # 90
                'video_play_cnt': list(self.data.iloc[:, 91]), # 91
                'reco_ban': [randint(0, 1) for _ in range(self.data.shape[0])],#暂时没有, mock吧
                ### 真正的label信息定义如下：
                'time_ratio': [],
                'time_reward': [],
                'time_idx_vec': [],
                'time_reward_start': [],
                'time_reward_range': [],
                'time_delta': [],
                
                'photo_ratio': [],
                'photo_reward': [],
                'photo_idx_vec': [],
                'photo_reward_start': [],
                'photo_reward_range': []
            },
            "not_final": list(self.data.iloc[:, 16]) #18
        }
        self.hash_size = hash_size

        self.user_type_mapping = {
            'Core': 1,
            'Gift': 2,
            'Potential_old': 3,
            'Potential_new': 4,
            'Long_old': 5,
            'Long_new': 6
        }
        #调用_process_labels
        self._process_labels()

    def __len__(self):
        return len(self.features['cur_features']['common_signs'])
    
    def _get_features(self, 
                        common_sign_list_str, 
                        common_slot_list_str, 
                        item_sign_list_str, 
                        item_slot_list_str,
                        user_slots,
                        item_slots,
                        attn1_slots,
                        attn2_slots):
        common_sign_list = self._parse_signs(common_sign_list_str)
        common_slot_list = self._parse_slots(common_slot_list_str)
        
        item_sign_list = self._parse_signs(item_sign_list_str)
        item_slot_list = self._parse_slots(item_slot_list_str)
        
        common_slot_dict = {}
        for slot, sign in zip(common_slot_list, common_sign_list):
            if slot in common_slot_dict:
                common_slot_dict[slot].append(sign)
            else:
                common_slot_dict[slot] = [sign]
        # key -value  slot:[signs]
        item_slot_dict = {slot: sign for slot, sign in zip(item_slot_list, item_sign_list)}
        
        cur_user_signs, cur_item_signs, cur_attn1_signs, cur_attn2_signs = [], [], [], []
        
        # 处理 cur_user_signs
        for slot in user_slots:
            signs = common_slot_dict.get(slot, [0])
            cur_user_signs.append(signs[0] if signs else 0)
        
        # 处理 cur_item_signs
        for slot in item_slots:
            cur_item_signs.append(item_slot_dict.get(slot, 0))
        
        def generate_attn_signs_per_slot(slots, slot_dict, max_length=50):
            attn_signs = []
            for slot in slots:
                signs = slot_dict.get(slot, [])
                # 如果slot数量不足50，填充0；如果超过50，截断
                if len(signs) >= max_length:
                    padded_signs = signs[:max_length]
                else:
                    padded_signs = signs + [0] * (max_length - len(signs))
                attn_signs.extend(padded_signs)
            return attn_signs
        
        # 处理 cur_attn1_signs 和 cur_attn2_signs
        cur_attn1_signs = generate_attn_signs_per_slot(attn1_slots, common_slot_dict, 50)
        cur_attn2_signs = generate_attn_signs_per_slot(attn2_slots, common_slot_dict, 50)
        
        return cur_user_signs, cur_item_signs, cur_attn1_signs, cur_attn2_signs
    def _process_labels(self):
        '''
        return [cur labels]:time_radio,time_reward,idx_vec,start,range,photo,,,,[nxt labels]10个
        '''
        w_live_time_ban = 0.34
        def calc_reward_info(time, reco_ban, ban_reward):
            ratio = 0.0
            reward = ban_reward
            max_thres = 1.0
            time_list = [0.0, 6.0, 15.0, 30.0, 60.0, 100.0, 600.0, 1200.0]
            reward_list = [0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.9, 1.0]
            bucket_num = len(time_list) - 1
            idx_vec = [0, bucket_num]
            reward_start = [0.0, 0.0]
            reward_range = [6.0, max_thres]
            if reco_ban == 1:
                for i in range(bucket_num):
                    if reward >= reward_list[i] and reward < reward_list[i + 1]:
                        reward_ratio = (reward - reward_list[i]) / (reward_list[i + 1] - reward_list[i])
                        ban_time = time_list[i] + reward_ratio * (time_list[i + 1] - time_list[i])
                        ratio = ban_time / max_thres
                        break
                #ban的情况下：  ratio=0, idx_vec=[0,8], reward_start = [0.0, 0.0] reward_range=[6.0, 1.0] reward = 0.0
                return ratio, idx_vec, reward_start, reward_range, reward
            #不ban
            for i in range(bucket_num):
                # i最大是6
                if time >= time_list[i] and time < time_list[i + 1]:
                    idx_vec = [i, bucket_num]
                    reward_start = [time_list[i], 0.0]
                    reward_range = [time_list[i + 1] - time_list[i], max_thres]
                    ratio = (time - time_list[i]) / (time_list[i + 1] - time_list[i])
                    reward = reward_list[i] + ratio * (reward_list[i + 1] - reward_list[i])
                    break
            # 假设 time=10s i=1 ,落在 1 2之间（0开始）
            # idx_vec = [1, 8] reward_start=[6.0, 0.0], reward_range = [15-6, 1.0], ratio = (10 -6) / (15 - 6); reward = 0.2 * ratio*(0.4 - 0.2)
            return ratio, idx_vec, reward_start, reward_range, reward
            
        def gen_time_reward(self):
            live_duration = (np.array(self.labels['cur_labels']['live_end_auto_watch']) + np.array(self.labels['cur_labels']['live_end_watch'])).tolist()
            reco_ban = self.labels['cur_labels']['reco_ban']
            nxt_live_duration = (np.array(self.labels['nxt_labels']['live_end_auto_watch']) + np.array(self.labels['nxt_labels']['live_end_watch'])).tolist()
            nxt_reco_ban = self.labels['nxt_labels']['reco_ban']
            
            ratio, time_idx_vec, time_reward_start, time_reward_range, time_reward, nxt_ratio, nxt_time_idx_vec, nxt_time_reward_start, nxt_time_reward_range, nxt_time_reward =[[0 for _ in range(len(reco_ban))] for _ in range(10)]
            # print('init: ', ratio,time_idx_vec)
            for i in range(len(reco_ban)):
                ratio[i], time_idx_vec[i], time_reward_start[i], time_reward_range[i], time_reward[i] = calc_reward_info(live_duration[i], reco_ban[i], 0.0)
                nxt_ratio[i], nxt_time_idx_vec[i], nxt_time_reward_start[i], nxt_time_reward_range[i], nxt_time_reward[i] = calc_reward_info(nxt_live_duration[i], nxt_reco_ban[i], 0.0)
            # 填充
            self.labels['cur_labels']['time_ratio'] = ratio
            self.labels['cur_labels']['time_reward'] = time_reward
            self.labels['cur_labels']['time_idx_vec'] = time_idx_vec
            self.labels['cur_labels']['time_reward_start'] = time_reward_start
            self.labels['cur_labels']['time_reward_range'] = time_reward_range
            
            self.labels['nxt_labels']['time_ratio'] = nxt_ratio
            self.labels['nxt_labels']['time_reward'] = nxt_time_reward
            self.labels['nxt_labels']['time_idx_vec'] = nxt_time_idx_vec
            self.labels['nxt_labels']['time_reward_start'] = nxt_time_reward_start
            self.labels['nxt_labels']['time_reward_range'] = nxt_time_reward_range
        
        def calc_photo_reward_info(time, reco_ban, ban_reward):       
            ratio = 0.0
            reward = ban_reward
            ban_time = 0.0
            max_thres = 25.0
            time_list = [0.0, 3.0, 10.0, 25.0, 50.0, 100.0, 600.0, 1200.0]
            reward_list = [0.0, 0.1, 0.3, 0.5, 0.6, 0.7, 0.9, 1.0]
            bucket_num = len(time_list) -1
            idx_vec = [2, bucket_num]
            reward_start = [10.0, 0.0]
            reward_range = [15.0, max_thres]
            
            if reco_ban == 1:
                for i in range(bucket_num):
                    if reward >= reward_list[i] and reward < reward_list[i + 1]:
                        reward_ratio = (reward - reward_list[i]) / (reward_list[i + 1] - reward_list[i])
                        ban_time = time_list[i] + reward_ratio * (time_list[i + 1] - time_list[i])
                        ratio = ban_time / max_thres
                        break
                return ratio, ban_time, idx_vec, reward_start, reward_range, reward
        
            for i in range(bucket_num):
                if time >= time_list[i] and time < time_list[i + 1]:
                    idx_vec = [i, bucket_num]
                    reward_start = [time_list[i], 0.0]
                    reward_range = [time_list[i + 1] - time_list[i], max_thres]
                    ratio = (time - time_list[i]) / (time_list[i + 1] - time_list[i])
                    reward = reward_list[i] + ratio * (reward_list[i + 1] - reward_list[i])
                    break
            return ratio, ban_time, idx_vec, reward_start, reward_range, reward
        
        def gen_photo_reward(self):
            reco_ban = self.labels['cur_labels']['reco_ban']
            nxt_reco_ban = self.labels['nxt_labels']['reco_ban']
            avg_photo_play_time = (np.array(self.labels['cur_labels']['video_play_time']) / (np.array(self.labels['cur_labels']['video_play_cnt']) + 1e-8)).tolist()
            
            nxt_avg_photo_play_time = (np.array(self.labels['nxt_labels']['video_play_time'])  / (np.array(self.labels['nxt_labels']['video_play_cnt']) + 1e-8)).tolist()
            time_reco_ban_reward = w_live_time_ban 
            
            ratio, photo_ban_time, photo_idx_vec, photo_reward_start, photo_reward_range, photo_reward, nxt_ratio, nxt_photo_ban_time, nxt_photo_idx_vec, nxt_photo_reward_start, nxt_photo_reward_range, nxt_photo_reward = [[0 for _ in range(len(reco_ban))] for _ in range(12)]
            for i in range(len(reco_ban)):
                ratio[i], photo_ban_time[i], photo_idx_vec[i], photo_reward_start[i], photo_reward_range[i], photo_reward[i] = calc_photo_reward_info(avg_photo_play_time[i], reco_ban[i], time_reco_ban_reward)
            
                nxt_ratio[i], nxt_photo_ban_time[i], nxt_photo_idx_vec[i], nxt_photo_reward_start[i], nxt_photo_reward_range[i], nxt_photo_reward[i] = calc_photo_reward_info(nxt_avg_photo_play_time[i], nxt_reco_ban[i], time_reco_ban_reward)
            # 填充
            self.labels['cur_labels']['photo_ratio'] = ratio
            self.labels['cur_labels']['photo_reward'] = photo_reward
            self.labels['cur_labels']['photo_idx_vec'] = photo_idx_vec
            self.labels['cur_labels']['photo_reward_start'] = photo_reward_start
            self.labels['cur_labels']['photo_reward_range'] = photo_reward_range
            
            self.labels['nxt_labels']['photo_ratio'] = nxt_ratio
            self.labels['nxt_labels']['photo_reward'] = nxt_photo_reward
            self.labels['nxt_labels']['photo_idx_vec'] = nxt_photo_idx_vec
            self.labels['nxt_labels']['photo_reward_start'] = nxt_photo_reward_start
            self.labels['nxt_labels']['photo_reward_range'] = nxt_photo_reward_range
            return photo_ban_time, nxt_photo_ban_time
            
        def gen_time_delta(self):
            photo_ban_time, nxt_photo_ban_time = gen_photo_reward(self)
            
            live_duration = (np.array(self.labels['cur_labels']['live_end_auto_watch'])  + np.array(self.labels['cur_labels']['live_end_watch'])).tolist() 
            nxt_live_duration = (np.array(self.labels['nxt_labels']['live_end_auto_watch']) + np.array(self.labels['nxt_labels']['live_end_watch'])).tolist()
            avg_photo_play_time = (np.array(self.labels['cur_labels']['video_play_time']) / (np.array(self.labels['cur_labels']['video_play_cnt']) + 1e-8)).tolist()
            nxt_avg_photo_play_time = (np.array(self.labels['nxt_labels']['video_play_time']) / (np.array(self.labels['nxt_labels']['video_play_cnt']) + 1e-8)).tolist()
            reco_ban = self.labels['cur_labels']['reco_ban']
            nxt_reco_ban = self.labels['nxt_labels']['reco_ban']
            time_diff = (np.array(live_duration) - np.array(avg_photo_play_time)).tolist()
            nxt_time_diff = (np.array(nxt_live_duration) - np.array(nxt_avg_photo_play_time)).tolist()
        
            if reco_ban == 1:
                time_diff = (-1 * np.array(photo_ban_time)).tolist()
        
            if nxt_reco_ban == 1:
                nxt_time_diff = (-1 * np.array(nxt_photo_ban_time)).tolist()
            self.labels['cur_labels']['time_delta'] = time_diff
            self.labels['nxt_labels']['time_delta'] = nxt_time_diff
        
        def process_second(self):
            self.labels['cur_labels']['live_end_auto_watch']  = (np.array(self.labels['cur_labels']['live_end_auto_watch']) / 1000).tolist()
            self.labels['cur_labels']['live_end_watch']  = (np.array(self.labels['cur_labels']['live_end_watch']) / 1000).tolist()
            self.labels['cur_labels']['video_play_time']  = (np.array(self.labels['cur_labels']['video_play_time']) / 1000).tolist()
            
            self.labels['nxt_labels']['live_end_auto_watch']  = (np.array(self.labels['nxt_labels']['live_end_auto_watch']) / 1000).tolist()
            self.labels['nxt_labels']['live_end_watch']  = (np.array(self.labels['nxt_labels']['live_end_watch']) / 1000).tolist()
            self.labels['nxt_labels']['video_play_time']  = (np.array(self.labels['nxt_labels']['video_play_time']) / 1000).tolist()
            
        process_second(self)
        gen_time_reward(self)
        gen_time_delta(self) #其中调用了gen_photo_reward
    
    def __getitem__(self, idx):
        # self.features['cur_features']['common_signs'][idx], self.features['cur_features']['common_slots'][idx], self.features['cur_features']['item_signs'][idx], self.features['cur_features']['item_slots'][idx],
        cur_user_slots =  [1000, 1001, 1008, 1009, 1020, 1004, 1005, 1006, 1007, 1200, 1201, 1202]
        cur_item_slots =  [100, 101, 102, 103, 105, 106, 107, 108, 109, 110]
        cur_attn1_slots = [1145, 1101, 1102, 1147]
        cur_attn2_slots = [1149, 1103, 1104, 1105, 1106]
        
        nxt_user_slots =  [1010, 1011, 1018, 1019, 1021, 1014, 1015, 1016, 1017, 1203, 1204, 1205]
        nxt_item_slots =  [112, 113, 114, 115, 117, 118, 119, 120, 121, 122]
        nxt_attn1_slots = [1146, 1123, 1124, 1148]
        nxt_attn2_slots = [1150, 1125, 1126, 1127, 1128]
        
        cur_user_signs, cur_item_signs, cur_attn1_signs, cur_attn2_signs = self._get_features(
            self.features['cur_features']['common_signs'][idx], 
            self.features['cur_features']['common_slots'][idx], 
            self.features['cur_features']['item_signs'][idx], 
            self.features['cur_features']['item_slots'][idx],
            cur_user_slots,
            cur_item_slots,
            cur_attn1_slots,
            cur_attn2_slots
        )
        nxt_user_signs, nxt_item_signs, nxt_attn1_signs, nxt_attn2_signs = self._get_features(
            self.features['nxt_features']['common_signs'][idx], 
            self.features['nxt_features']['common_slots'][idx], 
            self.features['nxt_features']['item_signs'][idx], 
            self.features['nxt_features']['item_slots'][idx],
            nxt_user_slots,
            nxt_item_slots,
            nxt_attn1_slots,
            nxt_attn2_slots
        )
        sample = {
            'cur_features': {
                'user_signs': cur_user_signs,
                'item_signs': cur_item_signs,
                'attn1_signs': cur_attn1_signs,
                'attn2_signs': cur_attn2_signs,
                'user_type': self._map_user_type(self.features['cur_features']['user_type'][idx])
            },
            'nxt_features': {
                'user_signs': nxt_user_signs,
                'item_signs': nxt_item_signs,
                'attn1_signs':nxt_attn1_signs,
                'attn2_signs':nxt_attn2_signs,
                'user_type': self._map_user_type(self.features['nxt_features']['user_type'][idx])
            },
            'cur_labels': {
                'live_end_auto_watch': self.labels['cur_labels']['live_end_auto_watch'][idx],
                'live_end_watch': self.labels['cur_labels']['live_end_watch'][idx],
                'video_play_time': self.labels['cur_labels']['video_play_time'][idx],  # 填充 video_play_time
                'video_play_cnt': self.labels['cur_labels']['video_play_cnt'][idx],  # 填充 video_play_cnt
                'reco_ban': self.labels['cur_labels']['reco_ban'][idx],  # 添加 reco_ban 标签
                'time_ratio': self.labels['cur_labels']['time_ratio'][idx] if len(self.labels['cur_labels']['time_ratio']) > 0 else None,
                'time_reward': self.labels['cur_labels']['time_reward'][idx] if len(self.labels['cur_labels']['time_reward']) > 0 else None,
                'time_idx_vec': self.labels['cur_labels']['time_idx_vec'][idx] if len(self.labels['cur_labels']['time_idx_vec']) > 0 else None,
                'time_reward_start': self.labels['cur_labels']['time_reward_start'][idx] if len(self.labels['cur_labels']['time_reward_start']) > 0 else None,
                'time_reward_range': self.labels['cur_labels']['time_reward_range'][idx] if len(self.labels['cur_labels']['time_reward_range']) > 0 else None,
                'time_delta': self.labels['cur_labels']['time_delta'][idx] if len(self.labels['cur_labels']['time_delta']) > 0 else None,
                'photo_ratio': self.labels['cur_labels']['photo_ratio'][idx] if len(self.labels['cur_labels']['photo_ratio']) > 0 else None,
                'photo_reward': self.labels['cur_labels']['photo_reward'][idx] if len(self.labels['cur_labels']['photo_reward']) > 0 else None,
                'photo_idx_vec': self.labels['cur_labels']['photo_idx_vec'][idx] if len(self.labels['cur_labels']['photo_idx_vec']) > 0 else None,
                'photo_reward_start': self.labels['cur_labels']['photo_reward_start'][idx] if len(self.labels['cur_labels']['photo_reward_start']) > 0 else None,
                'photo_reward_range': self.labels['cur_labels']['photo_reward_range'][idx] if len(self.labels['cur_labels']['photo_reward_range']) > 0 else None
            },
            'nxt_labels': {
                'live_end_auto_watch': self.labels['nxt_labels']['live_end_auto_watch'][idx],
                'live_end_watch': self.labels['nxt_labels']['live_end_watch'][idx],
                'video_play_time': self.labels['nxt_labels']['video_play_time'][idx],  # 填充 video_play_time
                'video_play_cnt': self.labels['nxt_labels']['video_play_cnt'][idx],  # 填充 video_play_cnt
                'reco_ban': self.labels['nxt_labels']['reco_ban'][idx],  # 添加 reco_ban 标签
                'time_ratio': self.labels['nxt_labels']['time_ratio'][idx] if len(self.labels['nxt_labels']['time_ratio']) > 0 else None,
                'time_reward': self.labels['nxt_labels']['time_reward'][idx] if len(self.labels['nxt_labels']['time_reward']) > 0 else None,
                'time_idx_vec': self.labels['nxt_labels']['time_idx_vec'][idx] if len(self.labels['nxt_labels']['time_idx_vec']) > 0 else None,
                'time_reward_start': self.labels['nxt_labels']['time_reward_start'][idx] if len(self.labels['nxt_labels']['time_reward_start']) > 0 else None,
                'time_reward_range': self.labels['nxt_labels']['time_reward_range'][idx] if len(self.labels['nxt_labels']['time_reward_range']) > 0 else None,
                'time_delta': self.labels['nxt_labels']['time_delta'][idx] if len(self.labels['nxt_labels']['time_delta']) > 0 else None,
                'photo_ratio': self.labels['nxt_labels']['photo_ratio'][idx] if len(self.labels['nxt_labels']['photo_ratio']) > 0 else None,
                'photo_reward': self.labels['nxt_labels']['photo_reward'][idx] if len(self.labels['nxt_labels']['photo_reward']) > 0 else None,
                'photo_idx_vec': self.labels['nxt_labels']['photo_idx_vec'][idx] if len(self.labels['nxt_labels']['photo_idx_vec']) > 0 else None,
                'photo_reward_start': self.labels['nxt_labels']['photo_reward_start'][idx] if len(self.labels['nxt_labels']['photo_reward_start']) > 0 else None,
                'photo_reward_range': self.labels['nxt_labels']['photo_reward_range'][idx] if len(self.labels['nxt_labels']['photo_reward_range']) > 0 else None
            },
            'not_final': self.labels['not_final'][idx]
        }
        return sample

    def _parse_signs(self, sign_str):
        '''
            处理sign，并且给他hash到一个范围内。
            根据我的统计：
                一共1000w sign。
                出现5次以上的100w.
                出现10次以上65w。
            结论：sign的分布非常的不均衡，所以直接给他hash映射就行了，小频率的sign不影响embedding layer的训练。
        '''
        # print(sign_str, type(sign_str))
        if not isinstance(sign_str, str):
            return []
        if pd.isna(sign_str) or sign_str.strip() == "[]":
            return []
        sign_str = sign_str.strip('[]')
        sign_list = sign_str.split(',')
        signs = [self._mod_sign(int(sign.strip())) for sign in sign_list if sign.strip().isdigit()]
        return signs

    def _parse_slots(self, slot_str):
        #处理slot，返回list
        if not isinstance(slot_str, str):
            return []
        if pd.isna(slot_str) or slot_str.strip() == "[]":
            return []
        slot_str = slot_str.strip('[]')
        slot_list = slot_str.split(',')
        return [int(slot.strip()) for slot in slot_list if slot.strip().isdigit()]
    def _map_user_type(self, user_type_str):
        try:
            res = self.user_type_mapping.get(user_type_str.strip(), 0)
        except Exception as e:
            # print(user_type_str, type(user_type_str))
            res = 6
        return res
    def _mod_sign(self, sign):
        return sign % self.hash_size
    def _check_data(self):
        for row_num in range(self.data.shape[0]):
            sign_str = self.data.iloc[row_num, 71]
            sign_str1 = self.data.iloc[row_num, 15]
            sign_str2 = self.data.iloc[row_num, 35]
            if not isinstance(sign_str, str):
                self.data.drop(row_num, inplace=True)
            if isinstance(sign_str1, int):
                self.data.drop(row_num, inplace=True)
            if isinstance(sign_str2, int):
                self.data.drop(row_num, inplace=True)

In [114]:
def collate_fn(batch):
    # 初始化各个batch的特征和标签
    batched_cur_features = {
        'user_signs': [],
        'item_signs': [],
        'attn1_signs': [],
        'attn2_signs': [],
        'user_type': [],
    }
    batched_nxt_features = {
        'user_signs': [],
        'item_signs': [],
        'attn1_signs': [],
        'attn2_signs': [],
        'user_type': [],
    }
    
    batched_cur_labels = {
        'live_end_auto_watch': [],
        'live_end_watch': [],
        'video_play_time': [],
        'video_play_cnt': [],
        'reco_ban': [],
        'time_ratio': [],
        'time_reward': [],
        'time_idx_vec': [],
        'time_reward_start': [],
        'time_reward_range': [],
        'time_delta':[],
        'photo_ratio': [],
        'photo_reward': [],
        'photo_idx_vec': [],
        'photo_reward_start': [],
        'photo_reward_range': []
    }
    batched_nxt_labels = {
        'live_end_auto_watch': [],
        'live_end_watch': [],
        'video_play_time': [],
        'video_play_cnt': [],
        'reco_ban': [],
        'time_ratio': [],
        'time_reward': [],
        'time_idx_vec': [],
        'time_reward_start': [],
        'time_reward_range': [],
        'time_delta':[],
        'photo_ratio': [],
        'photo_reward': [],
        'photo_idx_vec': [],
        'photo_reward_start': [],
        'photo_reward_range': []
    }
    not_final = []

    # 开始填充batch
    for sample in batch:
        # 填充 cur_features 和 nxt_features
        for key in batched_cur_features.keys():
            batched_cur_features[key].append(sample['cur_features'][key])
            batched_nxt_features[key].append(sample['nxt_features'][key])

        # 填充 cur_labels 和 nxt_labels
        for key in batched_cur_labels.keys():
            batched_cur_labels[key].append(sample['cur_labels'][key])
            batched_nxt_labels[key].append(sample['nxt_labels'][key])

        # 填充 not_final 标签
        not_final.append(sample['not_final'])

    return {
        'cur_features': batched_cur_features,
        'nxt_features': batched_nxt_features,
        'cur_labels': batched_cur_labels,
        'nxt_labels': batched_nxt_labels,
        'not_final': not_final
    }


# 重写base && Acotr && Critic

In [163]:
class SimpleDenseNetwork(nn.Module):
    def __init__(self, input_dim, units, top_no_act=False, act = nn.ELU()):
        # top_no_act 控制top层是不是有激活函数
        super(SimpleDenseNetwork, self).__init__()
        layers = []
        in_dim = input_dim
        layer_num = len(units)
        for i, unit in enumerate(units):
            layers.append(nn.Linear(in_dim, unit))
            if not (top_no_act and i == layer_num - 1):
                layers.append(act)
            in_dim = unit
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class InputNetwork(nn.Module):
    def __init__(self, embedding_dim, hash_size, hidden_layers):
        super(InputNetwork, self).__init__()
        self.embedding_layer = nn.Embedding(hash_size, embedding_dim)
        self.attn1 = nn.MultiheadAttention(embedding_dim * 4, num_heads=4)
        self.attn2 = nn.MultiheadAttention(embedding_dim * 5, num_heads=4)
        self.fc1 = nn.Linear(embedding_dim * 22, embedding_dim * 4)
        self.fc2 = nn.Linear(embedding_dim * 22, embedding_dim * 5)
        self.dense_layers = SimpleDenseNetwork(embedding_dim * 22 + (embedding_dim * 4) + (embedding_dim * 5), hidden_layers, top_no_act=True)
        
    def forward(self, features):
        batch_size = len(features['user_signs'])
        # features中包括 user_signs item_signs attn1_signs attn2_signs
        user_signs, item_signs, attn1_signs,attn2_signs = features["user_signs"],features["item_signs"],features["attn1_signs"],features["attn2_signs"]
        # List int --> embedding 
        user_embedding = self.embedding_layer(torch.LongTensor(user_signs)).reshape((batch_size, -1))  #[bs, flatten]
        item_embedding = self.embedding_layer(torch.LongTensor(item_signs)).reshape((batch_size, -1))  #[bs, flatten]
        user_item_embedding = torch.cat((user_embedding ,item_embedding), dim = 1)
        # shape变换，仔细检查。。。。。
        attn1_embeddings = self.embedding_layer(torch.LongTensor(attn1_signs)).view(batch_size, 4, 50, -1).permute(2, 0, 1, 3).reshape(50, batch_size, -1)
        attn2_embeddings = self.embedding_layer(torch.LongTensor(attn2_signs)).view(batch_size, 5, 50, -1).permute(2, 0, 1, 3).reshape(50, batch_size, -1)
        fc1_out = self.fc1(user_item_embedding)  
        fc2_out = self.fc2(user_item_embedding)
        query1 = fc1_out.unsqueeze(0)
        query2 = fc2_out.unsqueeze(0)
        # print(query1.shape, attn1_embeddings.shape)
        # 基于 attn1_signs 和 attn2_signs 创建 key_padding_mask
        attn1_signs_tensor = torch.LongTensor(attn1_signs).view(batch_size, 4, 50)  # [bs, 4, 50]
        attn2_signs_tensor = torch.LongTensor(attn2_signs).view(batch_size, 5, 50)  # [bs, 5, 50]

        # 创建 key_padding_mask（维度: [batch_size, seq_length]）
        attn1_key_padding_mask = (attn1_signs_tensor == 0).all(dim=1)  # [bs, 50]
        attn2_key_padding_mask = (attn2_signs_tensor == 0).all(dim=1)
        
        # print(attn2_signs_tensor, attn2_key_padding_mask)
        
        attn1_output, attn1_weights = self.attn1(query1, attn1_embeddings, attn1_embeddings)
        attn2_output, attn2_weights = self.attn2(query2, attn2_embeddings, attn2_embeddings)
        
        attn1_output = attn1_output.squeeze(0)
        attn2_output = attn2_output.squeeze(0)
        fused_features = torch.cat((user_item_embedding, attn1_output, attn2_output), dim=1)
        final_output = self.dense_layers(fused_features)
        # print('out:',fc1_out,fc2_out,attn1_embeddings,attn2_embeddings,  fused_features,  final_output)
        return final_output


In [164]:
class ActorNetwork(nn.Module):
    def __init__(self, input_emb_dim, actor_user_hidden_layers = [128, 63, 31, 2]):
        super(ActorNetwork, self).__init__()
        self.tower1 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.tower2 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.tower3 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.tower4 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.tower5 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.tower6 = SimpleDenseNetwork(input_emb_dim, actor_user_hidden_layers, top_no_act = True)
        self.ln = nn.LayerNorm(input_emb_dim)
        
    def forward(self, x, user_type_onehot):
        x = self.ln(x) 
        tower1_out = self.tower1(x).unsqueeze(2)
        tower2_out = self.tower2(x).unsqueeze(2)
        tower3_out = self.tower3(x).unsqueeze(2)
        tower4_out = self.tower4(x).unsqueeze(2)
        tower5_out = self.tower5(x).unsqueeze(2)
        tower6_out = self.tower6(x).unsqueeze(2)
        user_flag = user_type_onehot.unsqueeze(1)
        user_tower_list = torch.concat([tower1_out,tower2_out,tower3_out,tower4_out,tower5_out,tower6_out], dim =2) #[bs,dim, 6]
        return torch.mul(user_flag, user_tower_list).sum(dim = 2)

In [165]:
def new_huber_loss(label, pred, weight, alpha_val, delta_val):
    residual = torch.abs(label - pred)
    min_residual = torch.minimum(residual, torch.tensor(delta_val))
    max_neg_pred = torch.maximum(-pred, torch.tensor(0.0))    
    huber_loss = weight * (0.5 * torch.square(min_residual) + alpha_val * (residual - min_residual + max_neg_pred))
    return huber_loss

class StepCounter:
    def __init__(self):
        self.step = 0
    def get_step(self):
        return self.step
    def increment_step(self):
        self.step += 1
step_counter = StepCounter()

class SupervisedVisionNetwork(nn.Module):
    def __init__(self, input_emb_dim, sup_hidden_layers = [64, 32, 8]):
        super(SupervisedVisionNetwork, self).__init__()
        self.tower1 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        self.tower2 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        self.tower3 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        self.tower4 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        self.tower5 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        self.tower6 = SimpleDenseNetwork(input_emb_dim, sup_hidden_layers, top_no_act = True)
        
    def forward(self, x, user_type_onehot, idx_vec, reward_start, reward_range):
        tower1_out = self.tower1(x).unsqueeze(2)
        tower2_out = self.tower2(x).unsqueeze(2)
        tower3_out = self.tower3(x).unsqueeze(2)
        tower4_out = self.tower4(x).unsqueeze(2)
        tower5_out = self.tower5(x).unsqueeze(2)
        tower6_out = self.tower6(x).unsqueeze(2)
        user_flag = user_type_onehot.unsqueeze(1)
        user_tower_list = torch.concat([tower1_out,tower2_out,tower3_out,tower4_out,tower5_out,tower6_out], dim =2) #[bs,dim, 6]
        user_tower = torch.sigmoid(torch.mul(user_flag, user_tower_list).sum(dim = 2))  # 4 7
        # print('sup tower:', user_tower)
        # print(x,  user_type_onehot, idx_vec, reward_start, reward_range, user_flag)
        # print('------')
        # print(tower1_out,tower2_out,tower3_out,tower4_out,tower5_out,tower6_out)
        
        
        # pred_ratio = tf.gather(user_tower, idx_vec, batch_dims=1)
        pred_ratio = torch.gather(user_tower, 1, torch.LongTensor(idx_vec)) # 4 2
        # 这个监督网络预测出来是一个ratio，根据ratio和label反解出时长。
        pred_reward = torch.Tensor(reward_start) + pred_ratio * torch.Tensor(reward_range)
        return pred_ratio, pred_reward

class CriticNetwork(nn.Module):
    def __init__(self, input_emb_dim, critic_a_hidden_layers = [64, 32, 2]):
        super(CriticNetwork, self).__init__()
        self.ln = nn.LayerNorm(input_emb_dim)
        # 有俩supervised tower，一个是live 一个是 video。
        self.live_sup_tower = SupervisedVisionNetwork(input_emb_dim, sup_hidden_layers = [64, 32, 8])
        self.photo_sup_tower = SupervisedVisionNetwork(input_emb_dim, sup_hidden_layers = [64, 32, 8])
        
        # 用户tower
        self.tower1 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        self.tower2 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        self.tower3 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        self.tower4 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        self.tower5 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        self.tower6 = SimpleDenseNetwork(input_emb_dim, critic_a_hidden_layers, top_no_act = True)
        
    def forward(self, base_tower, labels, user_type_onehot):
        # 拿到Labels
        time_idx = labels["time_idx_vec"]
        photo_idx = labels["photo_idx_vec"]
        live_start = labels["time_reward_start"]
        live_range = labels["time_reward_range"]
        photo_start = labels["photo_reward_start"]
        photo_range = labels["photo_reward_range"]

        base_tower = self.ln(base_tower)
        #下面是过监督层。
        pred_live_ratio, pred_live_time = self.live_sup_tower(base_tower, user_type_onehot, time_idx, live_start, live_range)
        pred_photo_ratio, pred_photo_time = self.photo_sup_tower(base_tower, user_type_onehot, photo_idx, photo_start, photo_range)
        pred_time_delta = pred_live_time - pred_photo_time
        # print('live photo:', pred_live_ratio, pred_live_time)
        # 通过两个预测的时间差计算reward
        pred_reward = torch.sigmoid(0.1 * pred_time_delta) #时间差越大越好，越小，越接近负数，不能给reward
        
        #用户tower
        tower1_out = self.tower1(base_tower).unsqueeze(2)
        tower2_out = self.tower2(base_tower).unsqueeze(2)
        tower3_out = self.tower3(base_tower).unsqueeze(2)
        tower4_out = self.tower4(base_tower).unsqueeze(2)
        tower5_out = self.tower5(base_tower).unsqueeze(2)
        tower6_out = self.tower6(base_tower).unsqueeze(2)
        user_flag = user_type_onehot.unsqueeze(1)
        a_user_tower = torch.concat([tower1_out,tower2_out,tower3_out,tower4_out,tower5_out,tower6_out], dim =2)     #[bs,dim, 6]
        a_user_tower = torch.mul(user_flag, a_user_tower).sum(dim = 2)
        #最终的Q值计算
        pred_q_value = pred_reward.detach() + 0.9 * F.relu(a_user_tower)
        # print('pred: ', pred_reward, a_user_tower)
        #reward（时间差的sigmoid）, 时间差, live时间比例，live反解时长，photo时间比例，photo反解时长，q值（reward+0.99*）
        return pred_reward, pred_time_delta, pred_live_ratio, pred_live_time, pred_photo_ratio, pred_photo_time, pred_q_value

class TimeCriticNetwork(nn.Module):
    def __init__(self, base_output_dim):
        super(TimeCriticNetwork, self).__init__()
        # 这俩作用是取min
        self.critic1 = CriticNetwork(base_output_dim)
        self.critic2 = CriticNetwork(base_output_dim)
        
    def set_target_network(self, target_network):
        self.target_network = target_network
        
    def _get_cur_actions(self, reco_ban):
        actions = []
        for i in reco_ban:
            if i == 0:
                actions.append([1.0, 0.0])
            else:
                actions.append([0.0, 1.0])
        return torch.Tensor(actions)

    def forward(self, cur_x, nxt_x, cur_labels, nxt_labels, not_final, user_type_onehot):
        # define Hyper params
        alpha_time = 100.0
        delta_time = 0.1
        alpha_photo = 200.0
        delta_photo = 0.1
        alpha = 100.0
        delta = 0.1
        
        cur_q1_pred_reward, cur_q1_pred_time_delta, cur_q1_pred_live_time_ratio, cur_q1_pred_live_time, cur_q1_pred_photo_time_ratio, cur_q1_pred_photo_time, cur_q1_value = self.critic1(cur_x, cur_labels, user_type_onehot)
        cur_q2_pred_reward, cur_q2_pred_time_delta, cur_q2_pred_live_time_ratio, cur_q2_pred_live_time, cur_q2_pred_photo_time_ratio, cur_q2_pred_photo_time, cur_q2_value = self.critic2(cur_x, cur_labels, user_type_onehot)
        
        cur_q_value = torch.min(cur_q1_value, cur_q2_value)
        cur_q_argmax = torch.argmax(cur_q_value, dim = 1)
        
        #先得到cur_action，如果(cur_labels)reco_ban ==1 cur_action={0.0 1.0} 否则=={1.0, 0.0}
        cur_action = self._get_cur_actions(cur_labels['reco_ban'])
        
        # 计算 cur_q1_action_pred_reward 和 cur_q2_action_pred_reward
        cur_q1_action_pred_reward = torch.sum(cur_q1_pred_reward * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_reward = torch.sum(cur_q2_pred_reward * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_pred_time_delta 和 cur_q2_action_pred_time_delta
        cur_q1_action_pred_time_delta = torch.sum(cur_q1_pred_time_delta * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_time_delta = torch.sum(cur_q2_pred_time_delta * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_value 和 cur_q2_action_value
        cur_q1_action_value = torch.sum(cur_q1_value * cur_action, dim=1, keepdim=True)
        cur_q2_action_value = torch.sum(cur_q2_value * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_pred_live_time 和 cur_q2_action_pred_live_time
        cur_q1_action_pred_live_time = torch.sum(cur_q1_pred_live_time * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_live_time = torch.sum(cur_q2_pred_live_time * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_pred_photo_time 和 cur_q2_action_pred_photo_time
        cur_q1_action_pred_photo_time = torch.sum(cur_q1_pred_photo_time * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_photo_time = torch.sum(cur_q2_pred_photo_time * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_pred_live_time_ratio 和 cur_q2_action_pred_live_time_ratio
        cur_q1_action_pred_live_time_ratio = torch.sum(cur_q1_pred_live_time_ratio * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_live_time_ratio = torch.sum(cur_q2_pred_live_time_ratio * cur_action, dim=1, keepdim=True)
        
        # 计算 cur_q1_action_pred_photo_time_ratio 和 cur_q2_action_pred_photo_time_ratio
        cur_q1_action_pred_photo_time_ratio = torch.sum(cur_q1_pred_photo_time_ratio * cur_action, dim=1, keepdim=True)
        cur_q2_action_pred_photo_time_ratio = torch.sum(cur_q2_pred_photo_time_ratio * cur_action, dim=1, keepdim=True)
        
        nxt_q1_pred_reward, nxt_q1_pred_time_delta, nxt_q1_pred_live_time_ratio, nxt_q1_pred_live_time, nxt_q1_pred_photo_time_ratio, nxt_q1_pred_photo_time, nxt_q1_value = self.target_network.critic1(nxt_x, nxt_labels, user_type_onehot)
        
        nxt_q2_pred_reward, nxt_q2_pred_time_delta, nxt_q2_pred_live_time_ratio, nxt_q2_pred_live_time, nxt_q2_pred_photo_time_ratio, nxt_q2_pred_photo_time, nxt_q2_value = self.target_network.critic2(nxt_x, nxt_labels, user_type_onehot)
        
        nxt_q_value = torch.min(nxt_q1_value, nxt_q2_value)
        # print('nxt_q: ',nxt_q1_value, nxt_q2_value )
        # 确定性的q
        nxt_q_value, _ = torch.max(nxt_q_value, dim =1 , keepdim=True)
        # 实际上就是reward的label了
        cur_time_delta = torch.Tensor(cur_labels['time_delta'])
        nxt_time_delta = torch.Tensor(nxt_labels['time_delta'])
        cur_time_delta_reward = F.sigmoid(0.1 * cur_time_delta)
        nxt_time_delta_reward = F.sigmoid(0.1 * nxt_time_delta)
        q_label = cur_time_delta_reward.unsqueeze(1) + 0.9 * torch.Tensor(not_final).view(-1, 1) * nxt_q_value.detach()
        # print('q_label check: ', cur_time_delta_reward, nxt_q_value)
        nxt_action = self._get_cur_actions(nxt_labels['reco_ban'])  
        # 计算 nxt_q1_action_pred_reward 和 nxt_q2_action_pred_reward
        nxt_q1_action_pred_reward = torch.sum(nxt_q1_pred_reward * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_reward = torch.sum(nxt_q2_pred_reward * nxt_action, dim=1, keepdim=True)
        
        # 计算 nxt_q1_action_pred_time_delta 和 nxt_q2_action_pred_time_delta
        nxt_q1_action_pred_time_delta = torch.sum(nxt_q1_pred_time_delta * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_time_delta = torch.sum(nxt_q2_pred_time_delta * nxt_action, dim=1, keepdim=True)
        
        # 计算 nxt_q1_action_pred_live_time 和 nxt_q2_action_pred_live_time
        nxt_q1_action_pred_live_time = torch.sum(nxt_q1_pred_live_time * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_live_time = torch.sum(nxt_q2_pred_live_time * nxt_action, dim=1, keepdim=True)
        
        # 计算 nxt_q1_action_pred_photo_time 和 nxt_q2_action_pred_photo_time
        nxt_q1_action_pred_photo_time = torch.sum(nxt_q1_pred_photo_time * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_photo_time = torch.sum(nxt_q2_pred_photo_time * nxt_action, dim=1, keepdim=True)
        
        # 计算 nxt_q1_action_pred_live_time_ratio 和 nxt_q2_action_pred_live_time_ratio
        nxt_q1_action_pred_live_time_ratio = torch.sum(nxt_q1_pred_live_time_ratio * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_live_time_ratio = torch.sum(nxt_q2_pred_live_time_ratio * nxt_action, dim=1, keepdim=True)
        
        # 计算 nxt_q1_action_pred_photo_time_ratio 和 nxt_q2_action_pred_photo_time_ratio
        nxt_q1_action_pred_photo_time_ratio = torch.sum(nxt_q1_pred_photo_time_ratio * nxt_action, dim=1, keepdim=True)
        nxt_q2_action_pred_photo_time_ratio = torch.sum(nxt_q2_pred_photo_time_ratio * nxt_action, dim=1, keepdim=True)
        
        q1_live_time_reward_loss = new_huber_loss(torch.Tensor(cur_labels['time_ratio']), cur_q1_action_pred_live_time_ratio, 1.0, alpha_time, delta_time)
        q2_live_time_reward_loss = new_huber_loss(torch.Tensor(cur_labels['time_ratio']), cur_q2_action_pred_live_time_ratio, 1.0, alpha_time, delta_time)
        q1_photo_time_reward_loss = new_huber_loss(torch.Tensor(cur_labels['photo_ratio']), cur_q1_action_pred_photo_time_ratio, 1.0, alpha_photo, delta_photo)
        q2_photo_time_reward_loss = new_huber_loss(torch.Tensor(cur_labels['photo_ratio']), cur_q2_action_pred_photo_time_ratio, 1.0, alpha_photo, delta_photo)
        q1_reward_loss = new_huber_loss(cur_time_delta_reward, cur_q1_action_pred_reward, 10 * 1.0, alpha_time, delta_time)
        q2_reward_loss = new_huber_loss(cur_time_delta_reward, cur_q2_action_pred_reward, 10 * 1.0, alpha_time, delta_time)
        q1_loss = new_huber_loss(q_label, cur_q1_action_value, 1.0, alpha, delta)
        q2_loss = new_huber_loss(q_label, cur_q2_action_value, 1.0, alpha, delta)
        # print('q_etc: ',q1_loss, q2_loss, q_label)


        live_time_sup_loss = torch.sum(q1_live_time_reward_loss) + torch.sum(q2_live_time_reward_loss)
        photo_time_sup_loss = torch.sum(q1_photo_time_reward_loss) + torch.sum(q2_photo_time_reward_loss)
        time_delta_sup_loss = torch.sum(q1_reward_loss) + torch.sum(q2_reward_loss)
        sup_loss = live_time_sup_loss + photo_time_sup_loss + time_delta_sup_loss
        rl_loss = torch.sum(q1_loss) + torch.sum(q2_loss)

        return cur_q_value, live_time_sup_loss, photo_time_sup_loss, time_delta_sup_loss, sup_loss, rl_loss

class LiveRecoModel(nn.Module):
    def __init__(self, 
                 embedding_dim=32, 
                 hash_size=100000, 
                 hidden_layers=[256,128],
                 base_output_dim = 128
                 ):
        super(LiveRecoModel, self).__init__()
        self.input_network = InputNetwork(embedding_dim, hash_size, hidden_layers=hidden_layers)
        self.actor_network = ActorNetwork(base_output_dim)
        self.time_critic1 = TimeCriticNetwork(base_output_dim)
        self.time_critic2 = TimeCriticNetwork(base_output_dim)
        self.time_critic1.set_target_network(self.time_critic2)
        self.time_critic2.set_target_network(self.time_critic1)
    def _get_cur_actions(self, reco_ban):
        actions = []
        for i in reco_ban:
            if i == 0:
                actions.append([1.0, 0.0])
            else:
                actions.append([0.0, 1.0])
        return torch.Tensor(actions)
    
    def forward(self, input):
        cur_features, nxt_features, cur_labels, nxt_labels, not_final  = input['cur_features'],input['nxt_features'],input['cur_labels'],input['nxt_labels'], input['not_final']
        # print(not_final)
        cur_common_input_res = self.input_network(cur_features)
        
        user_type = cur_features['user_type'] # list: len = bs
        user_type_indices = torch.tensor(user_type, dtype=torch.long) - 1       # [1, 6]--->范围是0~5
        user_type_onehot = F.one_hot(user_type_indices, num_classes=6)  # [batch_size, user_type_num]
        cur_action_logits = self.actor_network(cur_common_input_res.detach(), user_type_onehot)
        cur_action_probs = F.softmax(cur_action_logits, dim=1)
        
        nxt_common_input_res = self.input_network(nxt_features)
        
        #交替训练 ----> 改成复制。
        time_q_value1, live_time_sup_loss1, photo_time_sup_loss1, time_delta_sup_loss1, sup_loss1, time_rl_loss1 = self.time_critic1(cur_common_input_res, nxt_common_input_res, cur_labels, nxt_labels, not_final, user_type_onehot)
        
        time_q_value2, live_time_sup_loss2, photo_time_sup_loss2, time_delta_sup_loss2, sup_loss2, time_rl_loss2 = self.time_critic2(cur_common_input_res, nxt_common_input_res, cur_labels, nxt_labels, not_final, user_type_onehot)
        
        # 外面定义step，实现交替训练的逻辑
        # 再把源代码复制过来
        cur_action = self._get_cur_actions(cur_labels['reco_ban'])
        cur_time_reward =torch.Tensor(cur_labels['time_reward']) 
        
        step_mod = step_counter.get_step() % 2
        
        time_q_value = (1.0 - step_mod) * time_q_value1 + step_mod * time_q_value2
        live_time_sup_loss = (1.0 - step_mod) * live_time_sup_loss1 + step_mod * live_time_sup_loss2
        photo_time_sup_loss = (1.0 - step_mod) * photo_time_sup_loss1 + step_mod * photo_time_sup_loss2
        time_delta_sup_loss = (1.0 - step_mod) * time_delta_sup_loss1 + step_mod * time_delta_sup_loss2
        sup_loss = (1.0 - step_mod) * sup_loss1 + step_mod * sup_loss2
        time_rl_loss = (1.0 - step_mod) * time_rl_loss1 + step_mod * time_rl_loss2
        # print('time_rl_loss: ', time_rl_loss1.item(), time_rl_loss2.item())
        
        cur_action_prob = torch.sum(cur_action_probs * cur_action, dim=1, keepdim=True)
        cur_log_action_prob = torch.log(cur_action_prob + 1e-10)
        
        time_actor_weight = 10.0
        ctr_loss_weight = 100.0
        cur_reg_weight = 0.0
        
        tot_q_value = time_actor_weight * time_q_value
        tot_q_probs = torch.softmax(tot_q_value.detach(), dim=1)
        tot_q_action_prob = torch.sum(tot_q_probs * cur_action, dim=1, keepdim=True)
        ones = torch.ones_like(tot_q_action_prob)
        ce_loss = torch.sum(ctr_loss_weight * F.cross_entropy(cur_action_logits, tot_q_probs.argmax(dim=1), reduction='none').view(-1, 1))
        reg_loss = torch.sum(cur_reg_weight * torch.abs(cur_action_prob - 0.5))
        actor_loss = ce_loss + reg_loss
        rl_loss = time_rl_loss
        critic_loss = sup_loss + rl_loss
        
        return cur_action, cur_time_reward, cur_action_probs, actor_loss, time_rl_loss, critic_loss
    
live_model = LiveRecoModel()
# live_model(batch_data)
txt_file_path= 'rl_model_train_data_for_torch_1010_mini.txt'
dataset = LiveRecommendationDataset(txt_file_path)
# print(f"Dataset Size: {len(dataset)}")

data_loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
# batch_data = next(iter(data_loader))
# print(f"Batch keys: {batch_data.keys()}")

epoch_num = 10
live_model = LiveRecoModel()
#定义优化器，lr等超参数
optimizer = torch.optim.Adam(live_model.parameters(), lr=1e-5)

for epoch in range(epoch_num):
    print(f'Epoch: {epoch}')
    for batch in data_loader:
        cur_action, cur_time_reward, cur_action_probs, actor_loss, time_rl_loss, critic_loss = live_model(batch)
        total_loss = actor_loss + critic_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        step_counter.increment_step()
        print(f'Batch Loss: {total_loss.item()}, actor_loss: {actor_loss}, time_rl_loss: {time_rl_loss}, critic_loss: {critic_loss}')

Epoch: 0
Batch Loss: 3199.940673828125, actor_loss: 279.6726379394531, time_rl_loss: 134.173095703125, critic_loss: 2920.26806640625
Batch Loss: 3730.95263671875, actor_loss: 293.7208251953125, time_rl_loss: 51.733116149902344, critic_loss: 3437.231689453125
Batch Loss: 2727.17578125, actor_loss: 259.8724060058594, time_rl_loss: 146.94723510742188, critic_loss: 2467.303466796875
Batch Loss: 4086.47216796875, actor_loss: 329.685546875, time_rl_loss: 133.55697631835938, critic_loss: 3756.78662109375
Batch Loss: 2958.85791015625, actor_loss: 264.7849426269531, time_rl_loss: 96.9612808227539, critic_loss: 2694.072998046875
Batch Loss: 2511.083251953125, actor_loss: 259.8772277832031, time_rl_loss: 81.1499252319336, critic_loss: 2251.2060546875
Batch Loss: 3046.380615234375, actor_loss: 255.58544921875, time_rl_loss: 67.17591857910156, critic_loss: 2790.795166015625
Batch Loss: 2644.88671875, actor_loss: 281.480712890625, time_rl_loss: 155.83526611328125, critic_loss: 2363.406005859375
Batc

Epoch: 0
q_etc:  tensor([[2.4080e+01, 1.0329e+00, 6.0137e-04, 2.4174e-03],
        [4.2644e+01, 1.9596e+01, 1.2036e+01, 1.5522e+01],
        [3.5200e+01, 1.2152e+01, 4.5925e+00, 8.0778e+00],
        [2.7725e+01, 4.6774e+00, 2.5295e-03, 6.0289e-01]],
       grad_fn=<MulBackward0>) tensor([[2.0492e+01, 2.7675e-03, 7.2067e-07, 5.6623e-04],
        [4.3376e+01, 2.0328e+01, 1.2769e+01, 1.6254e+01],
        [3.0931e+01, 7.8841e+00, 3.2432e-01, 3.8096e+00],
        [3.1868e+01, 8.8211e+00, 1.2613e+00, 4.7466e+00]],
       grad_fn=<MulBackward0>) tensor([[0.8176, 0.5871, 0.5115, 0.5464],
        [0.7440, 0.5135, 0.4379, 0.4728],
        [0.6642, 0.4337, 0.3581, 0.3930],
        [0.6755, 0.4450, 0.3694, 0.4043]])
q_etc:  tensor([[2.3161e+01, 1.1334e-01, 3.2475e-04, 1.8203e-03],
        [3.2227e+01, 9.1794e+00, 1.6196e+00, 5.1049e+00],
        [2.2247e+01, 4.2275e-03, 1.3371e-04, 1.3110e-03],
        [3.2661e+01, 9.6139e+00, 2.0541e+00, 5.5394e+00]],
       grad_fn=<MulBackward0>) tensor([[1.191

KeyboardInterrupt: 

In [162]:
import torch
import torch.nn.functional as F
from torch.nn import MultiheadAttention

# 设置随机种子以确保结果可重复
torch.manual_seed(0)

# 定义输入参数
embed_dim = 8  # 嵌入维度
num_heads = 2  # 注意力头的数量
seq_length = 5  # 序列长度
batch_size = 1  # 批次大小

# 创建查询、键和值的张量
query = torch.rand(seq_length, batch_size, embed_dim)
key = torch.rand(seq_length, batch_size, embed_dim)
value = torch.rand(seq_length, batch_size, embed_dim)

# 创建一个全掩码的key_padding_mask
key_padding_mask = torch.ones(batch_size, seq_length, dtype=torch.bool)

# 初始化多头自注意力层
attn = MultiheadAttention(embed_dim, num_heads)

# 计算注意力输出和注意力权重
attn_output, attn_weights = attn(query, key, value, key_padding_mask=key_padding_mask)

# 打印结果
print("Attention Output:\n", attn_output)
print("Attention Weights:\n", attn_weights)

Attention Output:
 tensor([[[nan, nan, nan, nan, nan, nan, nan, nan]],

        [[nan, nan, nan, nan, nan, nan, nan, nan]],

        [[nan, nan, nan, nan, nan, nan, nan, nan]],

        [[nan, nan, nan, nan, nan, nan, nan, nan]],

        [[nan, nan, nan, nan, nan, nan, nan, nan]]], grad_fn=<ViewBackward0>)
Attention Weights:
 tensor([[[nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan]]], grad_fn=<MeanBackward1>)
